In [17]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"C:\workSpace\DeepLearnin_sleep")
STAGE1_DIR = PROJECT_ROOT / "data" / "processed" / "pre_sleep_forecasting" / "design_c_stage1"
OUTPUT_DIR = STAGE1_DIR / "sequence_tensors"

DATASET_PATH = STAGE1_DIR / "pre_sleep_design_c_stage1_dataset_with_split.csv"
RAW_FEATURE_PATH = STAGE1_DIR / "pre_sleep_design_c_stage1_feature_columns.csv"
FINAL_FEATURE_PATH = STAGE1_DIR / "pre_sleep_design_c_stage1_final_feature_columns.csv"
REMOVED_FEATURE_PATH = STAGE1_DIR / "pre_sleep_design_c_stage1_zero_variance_removed_features.csv"
IMPUTER_PATH = STAGE1_DIR / "pre_sleep_design_c_stage1_median_imputer.joblib"
SCALER_PATH = STAGE1_DIR / "pre_sleep_design_c_stage1_standard_scaler.joblib"

WINDOWS = [3, 5, 7]
ID_COL = "participant_object_id"
TARGET_COL = "good_sleep_label"
SPLIT_COL = "pre_sleep_split"
TIME_COL = "sleep_start_datetime"

pd.read_csv(REMOVED_FEATURE_PATH, encoding="utf-8-sig").head()
pd.read_csv(REMOVED_FEATURE_PATH, encoding="utf-8-sig").columns.tolist()

['removed_feature', 'train_scaled_std']

In [18]:
df = pd.read_csv(DATASET_PATH, encoding="utf-8-sig")
raw_features = pd.read_csv(RAW_FEATURE_PATH, encoding="utf-8-sig")["feature"].tolist()
final_features = pd.read_csv(FINAL_FEATURE_PATH, encoding="utf-8-sig")["feature"].tolist()
removed_df = pd.read_csv(REMOVED_FEATURE_PATH, encoding="utf-8-sig")
removed_features = set(removed_df["removed_feature"].dropna().astype(str).tolist())


print(df.columns.tolist())
print(len(removed_features))
print(sorted(removed_features))

imputer = joblib.load(IMPUTER_PATH)
scaler = joblib.load(SCALER_PATH)

df[TIME_COL] = pd.to_datetime(df[TIME_COL])
df = df.sort_values([ID_COL, TIME_COL]).reset_index(drop=True)

print(df.shape)
print(df[SPLIT_COL].value_counts())
print(len(raw_features), len(removed_features), len(final_features))

['sleep_episode_id', 'participant_object_id', 'calendar_date', 'sleep_start_datetime', 'sleep_end_datetime', 'sleep_start_date', 'sleep_end_date', 'prediction_cutoff_datetime', 'good_sleep_label', 'cross_midnight', 'previous_day_feature_date', 'previous_day_daily_joined', 'previous_day_resting_hr_resting_heart_rate_mean', 'previous_day_resting_hr_error_mean', 'previous_day_resting_hr_record_count', 'previous_day_lightly_active_minutes_sum', 'previous_day_moderately_active_minutes_sum', 'previous_day_sedentary_minutes_sum', 'previous_day_very_active_minutes_sum', 'previous_day_steps_sum', 'previous_day_steps_record_count', 'previous_day_calories_sum', 'previous_day_calories_record_count', 'pre_sleep_window_start', 'pre_sleep_window_end', 'pre_sleep_window_hours', 'steps_pre_sleep_sum', 'steps_pre_sleep_record_count', 'steps_pre_sleep_active_record_count', 'steps_pre_sleep_last_3h_sum', 'steps_pre_sleep_last_1h_sum', 'calories_pre_sleep_sum', 'calories_pre_sleep_record_count', 'calories_

In [19]:
computed_final_features = [f for f in raw_features if f not in removed_features]

assert len(raw_features) == 70
assert len(removed_features) == 12
assert len(final_features) == 58
assert computed_final_features == final_features
assert len(raw_features) == imputer.n_features_in_
assert len(raw_features) == scaler.n_features_in_

In [20]:
raw_matrix = df[raw_features].copy()
imputed = imputer.transform(raw_matrix)
scaled_70 = scaler.transform(imputed).astype(np.float32)

keep_mask = np.array([f not in removed_features for f in raw_features])
scaled_58 = scaled_70[:, keep_mask]

assert scaled_58.shape[1] == 58
assert np.isfinite(scaled_58).all()

for idx, feature in enumerate(final_features):
    df[f"model_feature__{feature}"] = scaled_58[:, idx]

model_feature_cols = [f"model_feature__{f}" for f in final_features]

In [21]:
def build_sequences_for_window(data, window):
    rows = []

    X_list = []
    y_list = []
    participant_list = []
    target_time_list = []
    split_list = []

    for participant_id, group in data.groupby(ID_COL, sort=False):
        group = group.sort_values(TIME_COL).reset_index(drop=True)

        values = group[model_feature_cols].to_numpy(dtype=np.float32)
        targets = group[TARGET_COL].to_numpy(dtype=np.int64)
        times = group[TIME_COL].astype(str).to_numpy()
        splits = group[SPLIT_COL].to_numpy()

        for end_idx in range(window - 1, len(group)):
            sequence_split = splits[end_idx]

            X_list.append(values[end_idx - window + 1 : end_idx + 1])
            y_list.append(targets[end_idx])
            participant_list.append(participant_id)
            target_time_list.append(times[end_idx])
            split_list.append(sequence_split)

    return {
        "X": np.stack(X_list).astype(np.float32),
        "y": np.array(y_list, dtype=np.int64),
        "participant_object_id": np.array(participant_list),
        "target_sleep_start_datetime": np.array(target_time_list),
        "split": np.array(split_list),
        "feature_names": np.array(final_features),
    }

In [22]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary_rows = []

for window in WINDOWS:
    arrays = build_sequences_for_window(df, window)
    window_dir = OUTPUT_DIR / f"window_{window}"
    window_dir.mkdir(parents=True, exist_ok=True)

    for split_name in ["train", "validation", "test"]:
        mask = arrays["split"] == split_name

        output_path = window_dir / f"{split_name}.npz"
        np.savez_compressed(
            output_path,
            X=arrays["X"][mask],
            y=arrays["y"][mask],
            participant_object_id=arrays["participant_object_id"][mask],
            target_sleep_start_datetime=arrays["target_sleep_start_datetime"][mask],
            feature_names=arrays["feature_names"],
        )

        summary_rows.append({
            "window": window,
            "split": split_name,
            "samples": int(mask.sum()),
            "participants": int(pd.Series(arrays["participant_object_id"][mask]).nunique()),
            "time_steps": window,
            "features": len(final_features),
            "path": str(output_path.relative_to(PROJECT_ROOT)),
        })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_DIR / "sequence_tensor_summary.csv", index=False, encoding="utf-8-sig")
summary_df

,window,split,samples,participants,time_steps,features,path
0,3,train,2234,41,3,58,data\processed\pre_sleep_forecasting\design_c_...
1,3,validation,329,9,3,58,data\processed\pre_sleep_forecasting\design_c_...
2,3,test,853,14,3,58,data\processed\pre_sleep_forecasting\design_c_...
3,5,train,2153,40,5,58,data\processed\pre_sleep_forecasting\design_c_...
4,5,validation,312,8,5,58,data\processed\pre_sleep_forecasting\design_c_...
5,5,test,825,14,5,58,data\processed\pre_sleep_forecasting\design_c_...
6,7,train,2073,39,7,58,data\processed\pre_sleep_forecasting\design_c_...
7,7,validation,296,8,7,58,data\processed\pre_sleep_forecasting\design_c_...
8,7,test,797,14,7,58,data\processed\pre_sleep_forecasting\design_c_...
